# 06 -- Tweet & User Clustering

Two parallel clustering pipelines:

**Part A -- Tweet clustering** (content-based)
- Sentence embeddings via `all-MiniLM-L6-v2`
- Dimensionality reduction with UMAP
- Density-based clustering with HDBSCAN
- Cluster-personality overlap analysis (chi-squared, ARI, NMI)

**Part B -- Agent / user clustering** (behaviour-based)
- Feature engineering: tweet length, sentiment, emotion, lexical diversity,
  readability, emoji rate, reply coherence
- PCA reduction + GMM and HDBSCAN clustering
- Cluster profiling against held-out demographic and personality traits

**Input:** `DATA_DIR/tweets.csv`, `DATA_DIR/user_demo.csv`,
`DATA_DIR/sentiment_results.csv`, `DATA_DIR/coherence_results.csv`
**Output:** `DATA_DIR/tweet_clusters.csv`, `DATA_DIR/agent_clusters.csv`,
interactive HTML plots in `PLOT_DIR/`


## Dependencies

In [ ]:
# Run once -- safe to skip if already installed
# !pip install sentence-transformers umap-learn hdbscan textstat


## Imports

In [ ]:
import math
import re
import string
import warnings
from collections import Counter

import hdbscan
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import umap
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from scipy.stats import chi2_contingency
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from textstat import flesch_reading_ease

warnings.filterwarnings('ignore')

for pkg in ['wordnet', 'punkt', 'stopwords', 'punkt_tab', 'omw-1.4']:
    nltk.download(pkg, quiet=True)


## Config

Set `DATA_DIR` to the folder produced by the earlier notebooks.
All UMAP, HDBSCAN, and GMM hyperparameters live here.


In [ ]:
DATA_DIR = "data"   # <- change to your local path
PLOT_DIR = f"{DATA_DIR}/plots"

INPUT_TWEETS     = f"{DATA_DIR}/tweets.csv"
INPUT_USER_DEMO  = f"{DATA_DIR}/user_demo.csv"
INPUT_SENTIMENT  = f"{DATA_DIR}/sentiment_results.csv"
INPUT_COHERENCE  = f"{DATA_DIR}/coherence_results.csv"
OUTPUT_TWEETS    = f"{DATA_DIR}/tweet_clusters.csv"
OUTPUT_AGENTS    = f"{DATA_DIR}/agent_clusters.csv"

# Tweet clustering (Part A)
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST    = 0.1
HDBSCAN_MIN_CLUSTER_SIZE_TWEETS = 20
HDBSCAN_MIN_SAMPLES_TWEETS      = 3

# Agent clustering (Part B)
PCA_VARIANCE_THRESHOLD          = 0.80   # keep components explaining 80% of variance
GMM_N_COMPONENTS                = 4
HDBSCAN_MIN_CLUSTER_SIZE_AGENTS = 6
HDBSCAN_MIN_SAMPLES_AGENTS      = 2

PERSONALITY_TRAITS = ["ex", "oe", "co", "ag", "ne"]
DEMOGRAPHIC_TRAITS = ["gender", "leaning", "age", "education_level"]


---
## Part A -- Tweet content clustering

### Load and clean tweets

In [ ]:
raw = pd.read_csv(INPUT_TWEETS)
user_info = pd.read_csv(INPUT_USER_DEMO)
user_info = user_info.drop(0).reset_index(drop=True)  # row 0 is a header artefact
user_info = user_info[["id", "ex", "oe", "co", "ag", "ne",
                        "gender", "leaning", "age", "education_level"]]

# Merge personality & demographic features onto tweets
tweets = raw.rename(columns={"id": "tweet_id"}).merge(
    user_info.set_index("id"), left_on="user_id", right_index=True
)

print(f"Loaded {len(tweets):,} tweets from {tweets['user_id'].nunique():,} users")
tweets.head()


### Preprocessing

`clean_tweet` strips URLs, mentions, hashtags, and punctuation.
The raw text is preserved in `tweet`; `clean_tweet` is used only for embeddings.


In [ ]:
def clean_tweet(text: str) -> str:
    """Strip URLs, mentions, hashtags, punctuation; lowercase and collapse whitespace."""
    text = re.sub(r"http\S+|www\S+", "", str(text))
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    return re.sub(r"\s+", " ", text).strip().lower()


tweets["clean_tweet"] = tweets["tweet"].apply(clean_tweet)
tweets = tweets[tweets["clean_tweet"].str.strip() != ""].reset_index(drop=True)
print(f"After cleaning: {len(tweets):,} tweets retained")


### Sentence embeddings

`all-MiniLM-L6-v2` is a fast, effective model for short social-media texts.
Embeddings are L2-normalised so cosine similarity reduces to a dot product.


In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(
    tweets["clean_tweet"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"Embeddings shape: {embeddings.shape}")


### Dimensionality reduction -- UMAP

UMAP projects the 384-dimensional embedding space to 2D, preserving local
neighbourhood structure. `cosine` metric matches the embedding space geometry.


In [ ]:
reducer = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    n_components=2,
    min_dist=UMAP_MIN_DIST,
    metric="cosine",
    random_state=42,
)
embeddings_2d = reducer.fit_transform(embeddings)
tweets["x"] = embeddings_2d[:, 0]
tweets["y"] = embeddings_2d[:, 1]
print(f"UMAP complete -- shape: {embeddings_2d.shape}")


### Clustering -- HDBSCAN

HDBSCAN is a density-based algorithm that naturally handles noise (cluster `-1`).
`min_cluster_size` is the key parameter: increase it to get fewer, larger clusters.


In [ ]:
clusterer_tweets = hdbscan.HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE_TWEETS,
    min_samples=HDBSCAN_MIN_SAMPLES_TWEETS,
    metric="euclidean",
    cluster_selection_method="eom",
)
tweets["cluster"] = clusterer_tweets.fit_predict(embeddings_2d)

n_clusters = tweets["cluster"].nunique() - (1 if -1 in tweets["cluster"].values else 0)
n_noise    = (tweets["cluster"] == -1).sum()
print(f"HDBSCAN found {n_clusters} clusters | {n_noise:,} noise points ")


### Cluster labels -- TF-IDF top keywords

For each cluster, fit a TF-IDF model on that cluster's tweets and take the
words with the **lowest IDF** (most cluster-specific) as the label.


In [ ]:
cluster_labels = {}
for cid in sorted(tweets["cluster"].unique()):
    if cid == -1:
        cluster_labels[cid] = "Noise"
        continue
    cluster_tweets = tweets[tweets["cluster"] == cid]["clean_tweet"].tolist()
    tfidf = TfidfVectorizer(max_features=200, stop_words="english")
    tfidf.fit(cluster_tweets)
    scores   = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))
    top_words = sorted(scores, key=scores.get)[:4]   # lower IDF = more specific
    cluster_labels[cid] = f"Cluster {cid}: {', '.join(top_words)}"

tweets["cluster_label"] = tweets["cluster"].map(cluster_labels)

print("Cluster labels:")
for cid, label in cluster_labels.items():
    n = (tweets["cluster"] == cid).sum()
    print(f"  {label}  ({n:,} tweets)")


### Interactive scatter plot -- tweet clusters

Plotly scatter coloured by cluster label. Hover shows a 90-character tweet
preview. The HTML file can be opened in any browser for full interactivity.


In [ ]:
tweets["tweet_short"] = tweets["tweet"].str[:90] + "..."

fig = px.scatter(
    tweets, x="x", y="y",
    color="cluster_label",
    hover_data={"tweet_short": True, "x": False, "y": False, "cluster_label": False},
    title="Tweet Clusters (UMAP + HDBSCAN)",
    labels={"cluster_label": "Cluster"},
    color_discrete_sequence=px.colors.qualitative.Bold,
    template="plotly_white", width=950, height=650,
)
fig.update_traces(marker=dict(size=9, opacity=0.8, line=dict(width=0.5, color="white")))
fig.update_layout(title_font_size=16, legend=dict(itemsizing="constant"))

out = f"{PLOT_DIR}/tweet_clusters.html"
fig.write_html(out)
print(f"Interactive plot saved -> {out}")
fig.show()


### Personality overlay scatter plots

One plot per personality trait: colour = cluster, symbol = trait value.
This reveals whether any cluster is dominated by a particular personality type.


In [ ]:
default_symbols = [
    "circle", "x", "diamond", "cross", "square",
    "triangle-up", "triangle-down", "pentagon", "hexagon", "star",
]

for trait in PERSONALITY_TRAITS:
    unique_vals = sorted(tweets[trait].dropna().unique())
    symbol_map  = {v: default_symbols[i % len(default_symbols)]
                   for i, v in enumerate(unique_vals)}

    fig = px.scatter(
        tweets, x="x", y="y",
        color="cluster_label", symbol=trait, symbol_map=symbol_map,
        hover_data={"tweet_short": True, trait: True, "x": False, "y": False},
        title=f"Tweet Clusters -- colour: cluster | shape: {trait.upper()}",
        labels={"cluster_label": "Cluster", trait: trait.upper()},
        color_discrete_sequence=px.colors.qualitative.Bold,
        template="plotly_white", width=950, height=650,
    )
    fig.update_traces(marker=dict(size=9, opacity=0.85,
                                   line=dict(width=0.5, color="white")))
    out = f"{PLOT_DIR}/tweet_clusters_{trait}.html"
    fig.write_html(out)
    fig.show()
    print(f"  Saved -> {out}")


### Cluster x personality heatmaps

Annotated heatmaps show the count and row-percentage of each personality value
within each cluster. Helps identify personality signatures per cluster.


In [ ]:
df_clean_tweets = tweets[tweets["cluster"] != -1].copy()

for trait in PERSONALITY_TRAITS:
    ct = pd.crosstab(df_clean_tweets["cluster_label"], df_clean_tweets[trait])
    z_text = []
    for row in ct.values:
        total = row.sum()
        z_text.append([f"{v}<br>({v/total:.0%})" if total > 0 else "0" for v in row])

    fig = go.Figure(go.Heatmap(
        z=ct.values, x=ct.columns.tolist(), y=ct.index.tolist(),
        colorscale="Blues", colorbar=dict(title="Count"),
    ))
    for i, row in enumerate(z_text):
        for j, txt in enumerate(row):
            fig.add_annotation(x=ct.columns[j], y=ct.index[i], text=txt,
                               showarrow=False, font=dict(size=11, color="black"))
    fig.update_layout(
        title=f"Cluster x {trait.upper()} -- count + row %",
        xaxis_title=f"{trait.upper()}", yaxis_title="Cluster",
        template="plotly_white", width=750, height=500,
        font=dict(family="Inter, Arial, sans-serif"), margin=dict(l=120),
    )
    out = f"{PLOT_DIR}/heatmap_cluster_{trait}.html"
    fig.write_html(out)
    fig.show()
    print(f"  Saved -> {out}")


### Cluster-personality overlap: chi-squared, ARI, NMI

For each personality trait, tests whether the cluster partition and the trait
partition are statistically independent (chi-squared) and measures their
mutual information (ARI, NMI).


In [ ]:
for trait in PERSONALITY_TRAITS:
    print(f"\n--- {trait.upper()} " + "-" * 40)
    ct = pd.crosstab(df_clean_tweets["cluster_label"], df_clean_tweets[trait])
    chi2, p, dof, _ = chi2_contingency(ct)
    sig = "SIGNIFICANT (p < 0.05)" if p < 0.05 else "not significant (p >= 0.05)"
    print(f"  chi2={chi2:.3f}  p={p:.4f}  df={dof}  -> {sig}")

    unique_vals = df_clean_tweets[trait].dropna().unique()
    if len(unique_vals) == 2:
        le_map  = {v: i for i, v in enumerate(unique_vals)}
        encoded = df_clean_tweets[trait].map(le_map)
        ari = adjusted_rand_score(encoded, df_clean_tweets["cluster"])
        nmi = normalized_mutual_info_score(encoded, df_clean_tweets["cluster"])
        print(f"  ARI={ari:.3f}  NMI={nmi:.3f}")
    else:
        print(f"  ARI/NMI skipped (trait has {len(unique_vals)} values, need 2)")


---
## Part B -- Agent behaviour clustering

### Feature engineering

Aggregates tweet-level signals to one row per user (agent).
Features cover writing style (length, readability, punctuation, emoji),
sentiment consistency, lexical diversity, and reply coherence.


In [ ]:
# ── Base: user demographics ───────────────────────────────────────────────
agents = user_info.copy()

# ── Tweet length statistics ───────────────────────────────────────────────
tweets["length"] = tweets["tweet"].apply(len)
len_stats = (tweets.groupby("user_id")["length"]
             .agg(mean_len="mean", std_len="std")
             .fillna(0)
             .reset_index())
agents = agents.merge(len_stats, left_on="id", right_on="user_id", how="left")\
               .drop(columns="user_id", errors="ignore")

# ── Sentiment statistics (from notebook 02) ───────────────────────────────
sentiments = pd.read_csv(INPUT_SENTIMENT)
sent_stats = (sentiments.groupby("user_id")
              .agg(mean_sent=("roberta_score", "mean"),
                   std_sent=("roberta_score",  "std"),
                   mean_emo=("emotion_score",  "mean"),
                   std_emo=("emotion_score",   "std"))
              .fillna(0)
              .reset_index())
agents = agents.merge(sent_stats, left_on="id", right_on="user_id", how="left")\
               .drop(columns="user_id", errors="ignore")
agents[["mean_sent","std_sent","mean_emo","std_emo"]] = (
    agents[["mean_sent","std_sent","mean_emo","std_emo"]].fillna(0))

print(f"Agent feature table: {len(agents)} rows x {len(agents.columns)} columns")
agents.head()


### Lexical diversity -- MTLD and Shannon entropy

**MTLD** (Measure of Textual Lexical Diversity) is more robust than
type-token ratio for texts of different lengths.
**Shannon entropy** captures vocabulary unpredictability.
Both are computed on the concatenation of all tweets per user.


In [ ]:
stop_words  = set(stopwords.words('english'))
lemmatizer  = WordNetLemmatizer()


def _preprocess(text: str) -> str:
    """Lowercase, strip punctuation, remove stopwords, lemmatise."""
    if not isinstance(text, str):
        return ""
    text = re.sub(f"[{re.escape(string.punctuation)}]",
                  "", text.lower().strip())
    tokens = [lemmatizer.lemmatize(t)
              for t in word_tokenize(text)
              if t not in stop_words]
    return " ".join(tokens)


def mtld(tokens: list, ttr_threshold: float = 0.72) -> float:
    """
    Measure of Textual Lexical Diversity (MTLD).
    Averages forward and backward passes to reduce length sensitivity.
    Returns 0 for empty or single-token sequences.
    """
    def _one_pass(toks):
        factors, types, n = 0, set(), 0
        for tok in toks:
            n += 1
            types.add(tok)
            if len(types) / n <= ttr_threshold:
                factors += 1
                types, n = set(), 0
        if n:
            factors += (1 - len(types) / n) / (1 - ttr_threshold)
        return len(toks) / factors if factors else 0

    if len(tokens) < 2:
        return 0.0
    return (_one_pass(tokens) + _one_pass(tokens[::-1])) / 2


def shannon_entropy(tokens: list) -> float:
    """Shannon entropy (bits) over token frequencies."""
    counts = Counter(tokens)
    total  = len(tokens)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())


def lexical_features(text: str) -> dict:
    """Return MTLD, Shannon entropy, and token count for a text string."""
    tokens = word_tokenize(text)
    if len(tokens) < 5:
        return {"mtld": None, "entropy": None, "token_count": len(tokens)}
    return {"mtld": mtld(tokens), "entropy": shannon_entropy(tokens),
            "token_count": len(tokens)}


tweets["cleaned_tweet"] = tweets["tweet"].apply(_preprocess)
tweets["tokens"]        = tweets["cleaned_tweet"].apply(word_tokenize)

user_text        = tweets.groupby("user_id")["tweet"].apply(" ".join)
diversity_feats  = (user_text.apply(lexical_features)
                             .apply(pd.Series)
                             .reset_index())
agents = agents.merge(diversity_feats, left_on="id", right_on="user_id", how="left")\
               .drop(columns="user_id", errors="ignore")


### Linguistic fingerprints

LIWC-inspired features computed from raw tweet text:
pronoun ratio (I vs we/you), average sentence length, Flesch readability,
exclamation frequency, and ellipsis frequency.


In [ ]:
def extract_liwc_features(text: str) -> pd.Series:
    """
    Approximate LIWC-style features:
      pronoun_ratio      -- first-person / (second+third-person + eps)
      avg_sent_length    -- mean words per sentence
      readability_score  -- Flesch Reading Ease
      exclamation_freq   -- ! count / word count
      ellipsis_freq      -- ... count / word count
    """
    if not isinstance(text, str) or not text.strip():
        return pd.Series([0.0] * 5,
                         index=["pronoun_ratio", "avg_sent_length",
                                "readability_score", "exclamation_freq", "ellipsis_freq"])

    tokens     = word_tokenize(text.lower())
    total      = len(tokens) or 1
    sentences  = [s for s in re.split(r"[.!?]+", text) if s.strip()]

    i_pron     = sum(1 for w in tokens if w in {"i","me","my","mine","myself"})
    we_pron    = sum(1 for w in tokens if w in {"we","us","our","ours",
                                                 "you","your","yours"})
    return pd.Series({
        "pronoun_ratio":     i_pron / (we_pron + 1e-5),
        "avg_sent_length":   total / max(len(sentences), 1),
        "readability_score": flesch_reading_ease(text),
        "exclamation_freq":  text.count("!") / total,
        "ellipsis_freq":     text.count("...") / total,
    })


liwc_raw   = tweets["tweet"].apply(extract_liwc_features)
liwc_df    = pd.concat([tweets[["user_id"]], liwc_raw], axis=1)
user_fp    = liwc_df.groupby("user_id").mean().reset_index()
user_fp    = user_fp.merge(diversity_feats, on="user_id", how="left")

agents = agents.merge(user_fp, left_on="id", right_on="user_id", how="left")\
               .drop(columns="user_id", errors="ignore")


### Per-user tweet cosine similarity

The mean pairwise cosine similarity of a user's tweets measures how
thematically consistent their posting behaviour is. Users with low
self-similarity post on a wider range of topics.


In [ ]:
agent_cosine = {}
for uid in tweets["user_id"].unique():
    idx  = tweets[tweets["user_id"] == uid].index.tolist()
    embs = embeddings[idx]
    if len(embs) > 1:
        sim_mat = cosine_similarity(embs)
        up = np.triu_indices(len(embs), k=1)
        agent_cosine[uid] = float(np.mean(sim_mat[up]))
    else:
        agent_cosine[uid] = 1.0 if len(embs) == 1 else 0.0

cosine_df = pd.DataFrame(agent_cosine.items(),
                         columns=["id", "avg_tweet_cosine_similarity"])
agents = agents.merge(cosine_df, on="id", how="left")
agents["avg_tweet_cosine_similarity"] = (
    agents["avg_tweet_cosine_similarity"].fillna(0))
print("Per-user cosine similarity computed.")


### Emoji and punctuation rates

In [ ]:
EMOJI_RE = re.compile(
    "[\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251]+",
    flags=re.UNICODE,
)


def emoji_rate(text: str) -> float:
    n = len(str(text))
    return len(EMOJI_RE.findall(str(text))) / n if n else 0.0


def punctuation_rate(text: str) -> float:
    n = len(str(text))
    return sum(1 for ch in str(text) if ch in string.punctuation) / n if n else 0.0


tweets["emoji_rate"]       = tweets["tweet"].apply(emoji_rate)
tweets["punctuation_rate"] = tweets["tweet"].apply(punctuation_rate)

rate_stats = (tweets.groupby("user_id")
              .agg(mean_emoji_rate=("emoji_rate",       "mean"),
                   mean_punct_rate=("punctuation_rate", "mean"))
              .fillna(0)
              .reset_index())

agents = agents.merge(rate_stats, left_on="id", right_on="user_id", how="left")\
               .drop(columns="user_id", errors="ignore")


### Merge reply coherence

Average reply coherence (from `04_response_coherence.ipynb`) is added as a
behavioural feature. Users with no replies get the corpus mean imputed and
a binary `has_parent` indicator is added to capture this missingness explicitly.


In [ ]:
reply_sim = pd.read_csv(INPUT_COHERENCE)
reply_sim["coherence_score"] = (
    0.20 * reply_sim["cosine_similarity"] +
    0.35 * reply_sim["bs_f1"] +
    0.45 * reply_sim["cross_encoder_score"]
)
avg_coh = (reply_sim.groupby("user_id")["coherence_score"]
           .mean().reset_index()
           .rename(columns={"coherence_score": "avg_reply_coherence"}))

agents = agents.merge(avg_coh, left_on="id", right_on="user_id", how="left")\
               .drop(columns="user_id", errors="ignore")

agents["has_parent"] = agents["avg_reply_coherence"].notna().astype(int)
mean_coh = agents["avg_reply_coherence"].mean()
agents["avg_reply_coherence"] = agents["avg_reply_coherence"].fillna(mean_coh)

print(f"Agent feature matrix: {agents.shape}")
agents.head()


### Feature correlation heatmap

In [ ]:
# Select only the behavioural features (numerical) for the correlation matrix
behav_cols = [
    "mean_len", "std_len", "mean_sent", "std_sent", "mean_emo", "std_emo",
    "mtld", "entropy", "pronoun_ratio", "avg_sent_length", "readability_score",
    "exclamation_freq", "ellipsis_freq", "avg_tweet_cosine_similarity",
    "mean_emoji_rate", "mean_punct_rate", "avg_reply_coherence",
]
behav_cols = [c for c in behav_cols if c in agents.columns]

corr = agents[behav_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, cbar_kws={"shrink": 0.7})
plt.title("Behavioural feature correlation matrix")
plt.tight_layout()
plt.show()


### Prepare clustering matrix

Categorical traits are one-hot encoded. Rows with any remaining NaN
(e.g. users with too few tweets for diversity scores) are dropped.
The final matrix is the input for PCA and clustering.


In [ ]:
# Columns used as clustering features (exclude id and raw demographics)
cluster_features = [
    "mean_sent", "std_emo", "mtld", "pronoun_ratio",
    "mean_emoji_rate", "avg_reply_coherence", "has_parent",
]
cluster_features = [c for c in cluster_features if c in agents.columns]

agents_cluster_df = agents[cluster_features].copy().dropna()
print(f"Clustering matrix: {agents_cluster_df.shape} ")
print(f"Dropped {len(agents) - len(agents_cluster_df)} agents with missing features")


### PCA + GMM and HDBSCAN clustering

PCA retains enough components to explain `PCA_VARIANCE_THRESHOLD` of variance,
reducing noise before clustering. Both GMM and HDBSCAN are run so their
results can be compared.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(agents_cluster_df)

pca = PCA(n_components=PCA_VARIANCE_THRESHOLD)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA: {X_pca.shape[1]} components explain {PCA_VARIANCE_THRESHOLD:.0%} of variance")

# GMM
gmm = GaussianMixture(n_components=GMM_N_COMPONENTS,
                      covariance_type="full", random_state=42)
labels_gmm = gmm.fit_predict(X_pca)

# HDBSCAN
clusterer_agents = hdbscan.HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE_AGENTS,
    min_samples=HDBSCAN_MIN_SAMPLES_AGENTS,
    metric="euclidean", cluster_selection_method="eom",
)
labels_hdb = clusterer_agents.fit_predict(X_pca)

print(f"GMM   -> {len(set(labels_gmm))} clusters")
n_hdb = len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)
print(f"HDBSCAN -> {n_hdb} clusters | {(labels_hdb == -1).sum()} noise points")


### Visualise agent clusters

PCA 2D scatter for both GMM and HDBSCAN, plus per-feature histograms.

In [ ]:
pca_vis = PCA(n_components=2)
X_2d    = pca_vis.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, labels, title in [
    (axes[0], labels_gmm, f"GMM ({GMM_N_COMPONENTS} components)"),
    (axes[1], labels_hdb, "HDBSCAN"),
]:
    noise = labels == -1
    if noise.any():
        ax.scatter(X_2d[noise, 0], X_2d[noise, 1],
                   c="lightgrey", s=30, label="Noise", zorder=1)
    mask = ~noise
    sc = ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                    c=labels[mask], cmap="tab10", s=50, alpha=0.8, zorder=2)
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
plt.suptitle("Agent clusters (PCA 2D projection)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Per-feature distributions by HDBSCAN cluster
df_viz = agents_cluster_df.copy()
df_viz["cluster"] = labels_hdb
for col in cluster_features:
    plt.figure(figsize=(7, 3))
    for cl in sorted(df_viz["cluster"].unique()):
        plt.hist(df_viz[df_viz["cluster"] == cl][col],
                 alpha=0.55, label=f"Cluster {cl}", bins=15)
    plt.title(f"{col} by HDBSCAN cluster")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


### Interactive agent cluster scatter (Plotly)

In [ ]:
df_agent_viz = agents_cluster_df.copy()
df_agent_viz["id"]            = agents.loc[agents_cluster_df.index, "id"].values
df_agent_viz["mean_len"]      = agents.loc[agents_cluster_df.index, "mean_len"].values
df_agent_viz["PC1"]           = X_2d[:, 0]
df_agent_viz["PC2"]           = X_2d[:, 1]
df_agent_viz["Agent_Cluster"] = labels_hdb.astype(str)

fig = px.scatter(
    df_agent_viz, x="PC1", y="PC2",
    color="Agent_Cluster",
    hover_data={"id": True, "mean_len": ":.2f",
                "mean_sent": ":.2f", "PC1": False, "PC2": False},
    title="Agent Clusters (HDBSCAN on PCA-reduced behavioural features)",
    labels={"PC1": "PC 1", "PC2": "PC 2", "Agent_Cluster": "Cluster"},
    color_discrete_sequence=px.colors.qualitative.Plotly,
    template="plotly_white", width=950, height=650,
)
fig.update_traces(marker=dict(size=10, opacity=0.8,
                               line=dict(width=0.5, color="white")))
fig.update_layout(title_font_size=16, legend=dict(itemsizing="constant"))

out = f"{PLOT_DIR}/agent_clusters.html"
fig.write_html(out)
print(f"Interactive plot saved -> {out}")
fig.show()


### Cluster profiling against held-out traits

The clustering used only behavioural features. Here we check whether
the discovered clusters correlate with personality and demographics that
were held out -- a form of external validation.


In [ ]:
agents_with_clusters = agents.loc[agents_cluster_df.index].copy()
agents_with_clusters["Agent_Cluster"] = labels_hdb

held_out = PERSONALITY_TRAITS + DEMOGRAPHIC_TRAITS

for trait in held_out:
    if trait not in agents_with_clusters.columns:
        continue
    col = agents_with_clusters[trait]
    print(f"\n--- {trait.upper()} " + "-" * 40)

    if col.dtype == object or col.nunique() <= 10:
        ct = pd.crosstab(
            agents_with_clusters["Agent_Cluster"], col,
            normalize="index"
        ).round(3)
        print(ct.to_string())
        ct_raw = pd.crosstab(agents_with_clusters["Agent_Cluster"], col)
        if ct_raw.shape[0] > 1 and ct_raw.shape[1] > 1:
            chi2, p, dof, _ = chi2_contingency(ct_raw)
            sig = "SIGNIFICANT" if p < 0.05 else "not significant"
            print(f"  chi2={chi2:.3f}  p={p:.4f}  df={dof}  -> {sig}")
        unique_vals = col.dropna().unique()
        if len(unique_vals) == 2:
            le = {v: i for i, v in enumerate(unique_vals)}
            enc = col.dropna().map(le)
            clabels = agents_with_clusters.loc[enc.index, "Agent_Cluster"]
            print(f"  ARI={adjusted_rand_score(enc, clabels):.3f}  "
                  f"NMI={normalized_mutual_info_score(enc, clabels):.3f}")
    else:
        stats = agents_with_clusters.groupby("Agent_Cluster")[trait].describe()
        print(stats.round(3).to_string())


## Save outputs

In [ ]:
# Tweet clusters
tweets.to_csv(OUTPUT_TWEETS, index=False)
print(f"Tweet clusters -> {OUTPUT_TWEETS}")

# Agent clusters
agent_out = agents.loc[agents_cluster_df.index].copy()
agent_out["gmm_cluster"]   = labels_gmm
agent_out["hdbscan_cluster"] = labels_hdb
agent_out.to_csv(OUTPUT_AGENTS, index=False)
print(f"Agent clusters -> {OUTPUT_AGENTS}")
